<div style="display:flex; justify-content:space-between; align-items:center; width:100%; margin:8px 0 24px 0;">
  <div style="text-align:left;">
    <img src="https://www.ec-nantes.fr/medias/photo/logocn-rvb_1648479844750-png?ID_FICHE=178994&amp;INLINE=FALSE" alt="Centrale Nantes" style="height:72px; width:auto;">
  </div>
  <div style="text-align:right; font-size:18px; font-weight:600; color:#17324d; line-height:1.35;">
    MSc. CORO DASSIP
  </div>
</div>

<div style="border:2px solid #333; padding:14px 20px; margin:15px auto 25px auto; width:85%; max-width:900px; box-sizing:border-box; text-align:center;">
  <h1 style="margin:0;"><b>Image Processing Fundamentals — Implementation</b></h1>
</div>

## Setup — Environment and Configuration


In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
RNG = np.random.default_rng(42)

plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.titlesize"] = 11

## 1. Data and Output Paths


In [ ]:
def locate_lab_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "data").is_dir() and (candidate / "notebooks").is_dir():
            return candidate
    raise FileNotFoundError(
        "Could not locate the lab root containing data/ and notebooks/."
    )
LAB_DIR = locate_lab_root(Path.cwd())
DATA_DIR = LAB_DIR / "data"
OUTPUT_DIR = LAB_DIR / "outputs" / "figures"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
IMAGE_EXTENSIONS = {".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp"}
DATA_FILES = sorted(
    path for path in DATA_DIR.rglob("*")
    if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
)

assert DATA_FILES, f"No supported images found in: {DATA_DIR}"
def select_input_image(*keywords: str) -> Path:
    keywords = tuple(keyword.lower() for keyword in keywords)
    matches = [
        path for path in DATA_FILES
        if all(keyword in path.stem.lower() for keyword in keywords)
    ]
    if not matches:
        raise FileNotFoundError(
            f"No image matching {keywords} found in {DATA_DIR}"
        )
    return matches[0]
ROLE_PATTERNS = {
    "einstein": ("einstein",),
    "peppers": ("pepper",),
    "ballons": ("ballon",),
    "grass": ("grass",),
    "tower": ("tower",),
}

IMAGE_FILES = {
    role: select_input_image(*patterns)
    for role, patterns in ROLE_PATTERNS.items()
}

print("Lab directory :", LAB_DIR)
print("Data directory:", DATA_DIR)
print("Output folder :", OUTPUT_DIR)
print("Images discovered:", len(DATA_FILES))
print("Experiment roles :", len(IMAGE_FILES))

## 2. Sampling and Quantization Experiments


### Sampling Experiment


In [ ]:
x = np.linspace(0, 2 * np.pi, 256)
y = np.linspace(0, 2 * np.pi, 256)
xx, yy = np.meshgrid(x, y)

continuous_signal_like = (
    0.55
    + 0.25 * np.sin(2.0 * xx)
    + 0.20 * np.cos(3.0 * yy)
)
continuous_signal_like = np.clip(continuous_signal_like, 0.0, 1.0)
sampling_intervals = [1, 4, 8, 16]

fig, axes = plt.subplots(1, 4, figsize=(14, 3.4))

for ax, step in zip(axes, sampling_intervals):
    sampled_signal = continuous_signal_like[::step, ::step]
    ax.imshow(sampled_signal, cmap="gray", vmin=0, vmax=1, interpolation="nearest")
    ax.set_title(f"Sampling step = {step}\nshape = {sampled_signal.shape}")
    ax.axis("off")

fig.suptitle("Spatial Sampling")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "01_sampling.png", dpi=300, bbox_inches="tight")
plt.show()

### Quantization Experiment


In [ ]:
gradient = np.tile(np.linspace(0, 1, 512), (90, 1))
quantization_bit_depths = [1, 2, 4, 8]

fig, axes = plt.subplots(4, 1, figsize=(11, 6))

for ax, bits in zip(axes, quantization_bit_depths):
    levels = 2 ** bits
    quantized = np.round(gradient * (levels - 1)) / (levels - 1)

    ax.imshow(quantized, cmap="gray", vmin=0, vmax=1, aspect="auto")
    ax.set_title(f"{bits}-bit quantization → {levels} intensity levels")
    ax.axis("off")

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "02_quantization.png", dpi=300, bbox_inches="tight")
plt.show()

## 3. Pixel Coordinates and Array Representation


In [ ]:
toy_grayscale_image = np.array(
    [
        [0, 32, 64, 96, 128],
        [24, 56, 88, 120, 152],
        [48, 80, 112, 144, 176],
        [72, 104, 136, 168, 208],
        [96, 128, 160, 208, 255],
    ],
    dtype=np.uint8,
)

fig, ax = plt.subplots(figsize=(5, 4.5))
ax.imshow(toy_grayscale_image, cmap="gray", vmin=0, vmax=255)

for row in range(toy_grayscale_image.shape[0]):
    for col in range(toy_grayscale_image.shape[1]):
        ax.text(
            col,
            row,
            str(toy_grayscale_image[row, col]),
            ha="center",
            va="center",
            fontsize=8,
        )

ax.set_title("A grayscale image is a matrix of intensities")
ax.set_xlabel("x / column")
ax.set_ylabel("y / row")

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "03_grayscale_matrix.png", dpi=300, bbox_inches="tight")
plt.show()

print("Shape:", toy_grayscale_image.shape)
print("dtype:", toy_grayscale_image.dtype)
print("Pixel at row=2, column=3:", toy_grayscale_image[2, 3])

## 4. Image Representation Modes


In [ ]:
binary = np.zeros((120, 160), dtype=np.uint8)
binary[30:90, 45:120] = 255
grayscale = np.tile(
    np.linspace(0, 255, 160, dtype=np.uint8),
    (120, 1),
)

rgb = np.zeros((120, 160, 3), dtype=np.uint8)
rgb[:, :53] = [255, 0, 0]
rgb[:, 53:106] = [0, 255, 0]
rgb[:, 106:] = [0, 0, 255]

fig, axes = plt.subplots(1, 3, figsize=(11, 3.3))

axes[0].imshow(binary, cmap="gray", vmin=0, vmax=255)
axes[0].set_title(f"Binary\nshape={binary.shape}")

axes[1].imshow(grayscale, cmap="gray", vmin=0, vmax=255)
axes[1].set_title(f"Grayscale\nshape={grayscale.shape}")

axes[2].imshow(rgb)
axes[2].set_title(f"RGB\nshape={rgb.shape}")

for ax in axes:
    ax.axis("off")

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "04_image_types.png", dpi=300, bbox_inches="tight")
plt.show()

## 5. Load and Inspect Real Images


In [ ]:
images = {}

for name, path in IMAGE_FILES.items():
    images[name] = np.asarray(Image.open(path).convert("RGB"))

for name, image in images.items():
    print(
        f"{name:9s} | "
        f"shape={str(image.shape):16s} "
        f"dtype={image.dtype} "
        f"range=[{image.min()}, {image.max()}]"
    )

In [ ]:
fig, axes = plt.subplots(1, len(images), figsize=(16, 4))

for ax, (name, image) in zip(axes, images.items()):
    ax.imshow(image)
    ax.set_title(f"{name}\n{image.shape[1]}×{image.shape[0]}")
    ax.axis("off")

fig.suptitle("Reference Images")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "05_reference_images.png", dpi=300, bbox_inches="tight")
plt.show()

## 6. Dimensions, Resolution, Aspect Ratio, and Channels


In [ ]:
peppers_image = images["peppers"]

height, width, channels = peppers_image.shape

pixel_count = height * width

aspect_ratio = width / height

print(f"Height       : {height} pixels")
print(f"Width        : {width} pixels")
print(f"Channels     : {channels}")
print(f"Pixel count  : {pixel_count:,}")
print(f"Aspect ratio : {aspect_ratio:.3f}")

## 7. Data Types, Bit Depth, Dynamic Range, and Memory


In [ ]:
print("dtype:", peppers_image.dtype)
print("bytes per value:", peppers_image.dtype.itemsize)
print("array memory:", f"{peppers_image.nbytes:,} bytes")
print("array memory:", f"{peppers_image.nbytes / 1024**2:.3f} MiB")

uint8_info = np.iinfo(np.uint8)
print("uint8 range:", uint8_info.min, "to", uint8_info.max)

## 8. Display Scaling and Visualization Control


In [ ]:
low_contrast = np.linspace(90, 165, 256, dtype=np.uint8)
low_contrast = np.tile(low_contrast, (120, 1))

fig, axes = plt.subplots(1, 2, figsize=(9, 3.2))

axes[0].imshow(low_contrast, cmap="gray")
axes[0].set_title("Automatic display scaling")

axes[1].imshow(low_contrast, cmap="gray", vmin=0, vmax=255)
axes[1].set_title("Fixed display range: 0–255")

for ax in axes:
    ax.axis("off")

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "06_display_scaling.png", dpi=300, bbox_inches="tight")
plt.show()

## 9. Pixel Access and Safe Modification


In [ ]:
balloons_image = images["ballons"]

y, x = 140, 220
original_pixel = balloons_image[y, x].copy()

edited_balloons_image = balloons_image.copy()
radius = 6
edited_balloons_image[
    y - radius : y + radius + 1,
    x - radius : x + radius + 1,
] = [255, 0, 255]

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].imshow(balloons_image)
axes[0].scatter([x], [y], s=70, facecolors="none", edgecolors="yellow")
axes[0].set_title("Original + selected pixel")

axes[1].imshow(edited_balloons_image)
axes[1].set_title("Edited copy")

for ax in axes:
    ax.axis("off")

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "07_pixel_edit.png", dpi=300, bbox_inches="tight")
plt.show()

print("Selected coordinate (x, y):", (x, y))
print("Stored RGB value:", original_pixel)

## 10. Regions of Interest (ROI)


In [ ]:
tower_image = images["tower"]

y0, y1 = 120, 360
x0, x1 = 170, 390

region_of_interest = tower_image[y0:y1, x0:x1]

fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))

axes[0].imshow(tower_image)
axes[0].add_patch(
    plt.Rectangle(
        (x0, y0),
        x1 - x0,
        y1 - y0,
        fill=False,
        edgecolor="red",
        linewidth=2,
    )
)
axes[0].set_title("Full image and ROI")

axes[1].imshow(region_of_interest)
axes[1].set_title(f"ROI: {region_of_interest.shape[1]}×{region_of_interest.shape[0]}")

for ax in axes:
    ax.axis("off")

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "08_region_of_interest.png", dpi=300, bbox_inches="tight")
plt.show()

## 11. Pixel Neighborhoods


In [ ]:
einstein_rgb = images["einstein"]

einstein_grayscale_float = (
    0.299 * einstein_rgb[..., 0].astype(np.float32)
    + 0.587 * einstein_rgb[..., 1].astype(np.float32)
    + 0.114 * einstein_rgb[..., 2].astype(np.float32)
)
einstein_gray = np.clip(einstein_grayscale_float, 0, 255).astype(np.uint8)

y, x = 120, 120
center_patch_3x3 = einstein_gray[y - 1 : y + 2, x - 1 : x + 2]

print("Center pixel:", einstein_gray[y, x])
print("3×3 neighborhood:")
print(center_patch_3x3)

## 12. RGB Channel Decomposition


In [ ]:
red = peppers_image[..., 0]
green = peppers_image[..., 1]
blue = peppers_image[..., 2]

fig, axes = plt.subplots(1, 4, figsize=(14, 4))

axes[0].imshow(peppers_image)
axes[0].set_title("RGB")

axes[1].imshow(red, cmap="gray", vmin=0, vmax=255)
axes[1].set_title("Red channel")

axes[2].imshow(green, cmap="gray", vmin=0, vmax=255)
axes[2].set_title("Green channel")

axes[3].imshow(blue, cmap="gray", vmin=0, vmax=255)
axes[3].set_title("Blue channel")

for ax in axes:
    ax.axis("off")

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "09_rgb_channels.png", dpi=300, bbox_inches="tight")
plt.show()

print(
    "Channel means:",
    {
        "R": round(float(red.mean()), 2),
        "G": round(float(green.mean()), 2),
        "B": round(float(blue.mean()), 2),
    },
)

## 13. RGB and BGR Conventions


In [ ]:
rgb_pixel_example = peppers_image
bgr_channel_order_example = rgb_pixel_example[..., ::-1]

fig, axes = plt.subplots(1, 2, figsize=(9, 4))

axes[0].imshow(rgb_pixel_example)
axes[0].set_title("Correct RGB")

axes[1].imshow(bgr_channel_order_example)
axes[1].set_title("Channels reversed")

for ax in axes:
    ax.axis("off")

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "10_rgb_bgr.png", dpi=300, bbox_inches="tight")
plt.show()

## 14. RGB-to-Grayscale Conversion


In [ ]:
def convert_rgb_to_grayscale(rgb_image: np.ndarray) -> np.ndarray:
    rgb_float = rgb_image.astype(np.float32)

    gray = (
        0.299 * rgb_float[..., 0]
        + 0.587 * rgb_float[..., 1]
        + 0.114 * rgb_float[..., 2]
    )

    return np.clip(gray, 0, 255).astype(np.uint8)
einstein_rgb = images["einstein"]
einstein_gray = convert_rgb_to_grayscale(einstein_rgb)

fig, axes = plt.subplots(1, 2, figsize=(9, 4))

axes[0].imshow(einstein_rgb)
axes[0].set_title(f"RGB shape: {einstein_rgb.shape}")

axes[1].imshow(einstein_gray, cmap="gray", vmin=0, vmax=255)
axes[1].set_title(f"Grayscale shape: {einstein_gray.shape}")

for ax in axes:
    ax.axis("off")

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "11_rgb_to_grayscale.png", dpi=300, bbox_inches="tight")
plt.show()

## 15. Image Statistics


In [ ]:
grass_grayscale_image = convert_rgb_to_grayscale(images["grass"])

statistics = {
    "min": int(grass_grayscale_image.min()),
    "max": int(grass_grayscale_image.max()),
    "mean": float(grass_grayscale_image.mean()),
    "median": float(np.median(grass_grayscale_image)),
    "std": float(grass_grayscale_image.std()),
    "p05": float(np.percentile(grass_grayscale_image, 5)),
    "p95": float(np.percentile(grass_grayscale_image, 95)),
}

for name, value in statistics.items():
    print(f"{name:>6s}: {value:.3f}" if isinstance(value, float) else f"{name:>6s}: {value}")

## 16. Intensity Histograms


In [ ]:
balloons_grayscale_image = convert_rgb_to_grayscale(images["ballons"])

counts, bin_edges = np.histogram(
    balloons_grayscale_image.ravel(),
    bins=256,
    range=(0, 256),
)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].imshow(balloons_grayscale_image, cmap="gray", vmin=0, vmax=255)
axes[0].set_title("Grayscale image")
axes[0].axis("off")

axes[1].plot(np.arange(256), counts)
axes[1].set_title("Intensity histogram")
axes[1].set_xlabel("Intensity")
axes[1].set_ylabel("Pixel count")
axes[1].set_xlim(0, 255)

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "12_intensity_histogram.png", dpi=300, bbox_inches="tight")
plt.show()

print("Histogram count:", counts.sum())
print("Number of image pixels:", balloons_grayscale_image.size)

In [ ]:
shuffled_pixel_image = balloons_grayscale_image.ravel().copy()
RNG.shuffle(shuffled_pixel_image)
shuffled_pixel_image = shuffled_pixel_image.reshape(balloons_grayscale_image.shape)

hist_original, _ = np.histogram(balloons_grayscale_image.ravel(), bins=256, range=(0, 256))
hist_shuffled, _ = np.histogram(shuffled_pixel_image.ravel(), bins=256, range=(0, 256))

print("Histograms identical:", np.array_equal(hist_original, hist_shuffled))

fig, axes = plt.subplots(1, 2, figsize=(9, 4))

axes[0].imshow(balloons_grayscale_image, cmap="gray", vmin=0, vmax=255)
axes[0].set_title("Original")

axes[1].imshow(shuffled_pixel_image, cmap="gray", vmin=0, vmax=255)
axes[1].set_title("Pixels shuffled")

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()

## 17. Dynamic Range and Min-Max Normalization


In [ ]:
def normalize_minmax(gray_image: np.ndarray) -> np.ndarray:
    image_float = gray_image.astype(np.float32)

    minimum = image_float.min()
    maximum = image_float.max()
    if maximum == minimum:
        return np.zeros_like(gray_image)

    normalized = (image_float - minimum) / (maximum - minimum)
    normalized *= 255.0

    return np.clip(normalized, 0, 255).astype(np.uint8)
source_grayscale_image = convert_rgb_to_grayscale(images["ballons"])
low_contrast = 90 + (source_grayscale_image.astype(np.float32) / 255.0) * 76
low_contrast = np.clip(low_contrast, 0, 255).astype(np.uint8)

normalized = normalize_minmax(low_contrast)

fig, axes = plt.subplots(2, 2, figsize=(10, 7))

axes[0, 0].imshow(low_contrast, cmap="gray", vmin=0, vmax=255)
axes[0, 0].set_title("Low-contrast image")
axes[0, 0].axis("off")

axes[0, 1].hist(low_contrast.ravel(), bins=256, range=(0, 256))
axes[0, 1].set_title("Before normalization")
axes[0, 1].set_xlabel("Intensity")

axes[1, 0].imshow(normalized, cmap="gray", vmin=0, vmax=255)
axes[1, 0].set_title("After min-max normalization")
axes[1, 0].axis("off")

axes[1, 1].hist(normalized.ravel(), bins=256, range=(0, 256))
axes[1, 1].set_title("After normalization")
axes[1, 1].set_xlabel("Intensity")

fig.tight_layout()
fig.savefig(
    OUTPUT_DIR / "13_dynamic_range_normalization.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()

print("Before range:", int(low_contrast.min()), "to", int(low_contrast.max()))
print("After range :", int(normalized.min()), "to", int(normalized.max()))

## 18. `uint8` Arithmetic, Overflow, Clipping, and Floating Point


In [ ]:
value = np.array([250], dtype=np.uint8)
unsafe = value + np.array([20], dtype=np.uint8)
safe_float = value.astype(np.float32) + 20.0
safe_uint8 = np.clip(safe_float, 0, 255).astype(np.uint8)

print("Original value :", value[0])
print("Unsafe result  :", unsafe[0])
print("Safe result    :", safe_uint8[0])

## 19. Noise Model Simulation


In [ ]:
base_grayscale_image = einstein_gray
gaussian_noise_samples = RNG.normal(0.0, 20.0, size=base_grayscale_image.shape)
gaussian_noisy_image = np.clip(
    base_grayscale_image.astype(np.float32) + gaussian_noise_samples,
    0,
    255,
).astype(np.uint8)
salt_and_pepper_noisy_image = base_grayscale_image.copy()

probability = 0.03
salt_pepper_random_map = RNG.random(base_grayscale_image.shape)
salt_and_pepper_noisy_image[salt_pepper_random_map < probability / 2] = 0
salt_and_pepper_noisy_image[salt_pepper_random_map > 1 - probability / 2] = 255
scaled = base_grayscale_image.astype(np.float32) / 255.0
poisson_noisy_image = RNG.poisson(scaled * 30.0) / 30.0
poisson_noisy_image = np.clip(poisson_noisy_image * 255.0, 0, 255).astype(np.uint8)
speckle_noise = RNG.normal(0.0, 0.18, size=base_grayscale_image.shape)
speckle_noisy_image = base_grayscale_image.astype(np.float32) * (1.0 + speckle_noise)
speckle_noisy_image = np.clip(speckle_noisy_image, 0, 255).astype(np.uint8)

fig, axes = plt.subplots(1, 5, figsize=(16, 3.6))

examples = [
    ("Original", base_grayscale_image),
    ("Gaussian", gaussian_noisy_image),
    ("Salt & pepper", salt_and_pepper_noisy_image),
    ("Poisson", poisson_noisy_image),
    ("Speckle", speckle_noisy_image),
]

for ax, (title, image) in zip(axes, examples):
    ax.imshow(image, cmap="gray", vmin=0, vmax=255)
    ax.set_title(title)
    ax.axis("off")

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "14_noise_models.png", dpi=300, bbox_inches="tight")
plt.show()

## 20. Image Comparison Metrics


In [ ]:
def compute_image_quality_metrics(reference: np.ndarray, test: np.ndarray) -> dict:
    if reference.shape != test.shape:
        raise ValueError("Images must have identical shapes.")

    reference_f = reference.astype(np.float64)
    test_f = test.astype(np.float64)

    difference = reference_f - test_f
    mae = np.mean(np.abs(difference))

    mse = np.mean(difference ** 2)

    rmse = np.sqrt(mse)
    if mse == 0:
        psnr = np.inf
    else:
        psnr = 10.0 * np.log10((255.0 ** 2) / mse)

    return {
        "MAE": mae,
        "MSE": mse,
        "RMSE": rmse,
        "PSNR": psnr,
    }
noise_results = {
    "Gaussian": compute_image_quality_metrics(base_grayscale_image, gaussian_noisy_image),
    "Salt & pepper": compute_image_quality_metrics(base_grayscale_image, salt_and_pepper_noisy_image),
    "Poisson": compute_image_quality_metrics(base_grayscale_image, poisson_noisy_image),
    "Speckle": compute_image_quality_metrics(base_grayscale_image, speckle_noisy_image),
}

for noise_name, metrics in noise_results.items():
    print(noise_name)
    for metric_name, value in metrics.items():
        print(f"  {metric_name:>4s}: {value:.4f}")

## 21. Lossless vs Lossy Image Encoding


In [ ]:
example = images["peppers"]
png_path = OUTPUT_DIR / "15_saved_example.png"

jpg_path = OUTPUT_DIR / "15_saved_example.jpg"

Image.fromarray(example).save(png_path)

Image.fromarray(example).save(jpg_path, quality=75)

png_reload = np.asarray(Image.open(png_path).convert("RGB"))
jpg_reload = np.asarray(Image.open(jpg_path).convert("RGB"))

png_metrics = compute_image_quality_metrics(example, png_reload)
jpg_metrics = compute_image_quality_metrics(example, jpg_reload)

print("PNG reload MSE :", png_metrics["MSE"])
print("JPEG reload MSE:", jpg_metrics["MSE"])
print("PNG path :", png_path)
print("JPEG path:", jpg_path)

## 22. Standard Image Inspection Workflow


In [ ]:
def summarize_image_properties(name: str, image: np.ndarray) -> None:

    print(f"Name       : {name}")
    print(f"Shape      : {image.shape}")
    print(f"Dimensions : {image.ndim}")
    print(f"dtype      : {image.dtype}")
    print(f"Min / max  : {image.min()} / {image.max()}")
    print(f"Mean       : {image.mean():.3f}")
    print(f"Memory     : {image.nbytes:,} bytes")
summarize_image_properties("peppers", peppers_image)

## 23. Validation Checks


In [ ]:
assert peppers_image.ndim == 3
assert peppers_image.shape[2] == 3
assert einstein_gray.ndim == 2

assert peppers_image.dtype == np.uint8
assert einstein_gray.dtype == np.uint8

assert counts.sum() == balloons_grayscale_image.size

assert normalized.min() == 0
assert normalized.max() == 255

self_metrics = compute_image_quality_metrics(einstein_gray, einstein_gray)
assert self_metrics["MAE"] == 0
assert self_metrics["MSE"] == 0
assert self_metrics["RMSE"] == 0
assert np.isinf(self_metrics["PSNR"])

assert region_of_interest.shape[0] == (y1 - y0)
assert region_of_interest.shape[1] == (x1 - x0)

print("All fundamental validation checks passed.")
REQUIRED_OUTPUTS = [
    "01_sampling.png",
    "02_quantization.png",
    "03_grayscale_matrix.png",
    "04_image_types.png",
    "05_reference_images.png",
    "06_display_scaling.png",
    "07_pixel_edit.png",
    "08_region_of_interest.png",
    "09_rgb_channels.png",
    "10_rgb_bgr.png",
    "11_rgb_to_grayscale.png",
    "12_intensity_histogram.png",
    "13_dynamic_range_normalization.png",
    "14_noise_models.png",
    "15_saved_example.png",
    "15_saved_example.jpg",
]
missing_outputs = [
    output_name
    for output_name in REQUIRED_OUTPUTS
    if not (OUTPUT_DIR / output_name).exists()
]
if missing_outputs:
    raise FileNotFoundError(
        "Missing outputs: " + ", ".join(missing_outputs)
    )

print(f"Output validation passed: {len(REQUIRED_OUTPUTS)} files.")
